In [1]:
import numpy as np
import torch
import sys
from torch.utils.data import DataLoader
!rm -r /root/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git clone https://github.com/KirillVishnyakov/Architectural-Biases-in-Time-Series-Anomaly-Detection
!git -C /root/Architectural-Biases-in-Time-Series-Anomaly-Detection log --oneline -1

rm: cannot remove '/root/Architectural-Biases-in-Time-Series-Anomaly-Detection': No such file or directory
Cloning into 'Architectural-Biases-in-Time-Series-Anomaly-Detection'...
remote: Enumerating objects: 327, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 327 (delta 63), reused 87 (delta 31), pack-reused 207 (from 1)
Receiving objects: 100% (327/327), 2.93 MiB | 14.73 MiB/s, done.
Resolving deltas: 100% (174/174), done.
acbf5b6 (HEAD -> main, origin/main, origin/HEAD) revin layer before feature mixer for cloud


In [2]:
sys.path.append("/root/Architectural-Biases-in-Time-Series-Anomaly-Detection")
import config as config
config.init("/mnt/data/data.csv", "/root/Architectural-Biases-in-Time-Series-Anomaly-Detection/checkpoint_dir")
import transformer_encoder
import dataset
import train
device = None
if torch.cuda.is_available():
    device = torch.device("cuda")
print(torch.__version__)

2.8.0+cu129


### Set random states for reproducibility

In [3]:
seed = 432
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
train_dataset = dataset.forecasting_Dataset(device = device, window_size = 512, horizon = 50, start = 0, end = 1000000 - 200000)
test_dataset = dataset.forecasting_Dataset(device = device, window_size = 512, horizon = 50, scaler = train_dataset.scaler, start = 1000000 - 200000, end = 1000000, train = False)
model = transformer_encoder.patch_transformer_encoder(lookback_window = 512, forecast_horizon = 50, d_model = 512, nhead = 16).to(device)
model = torch.compile(model)

64


In [5]:
best_model_wts, best_loss, avg_train_loss = train.fit_forecaster(device, model, "blind_transformer", train_dataset, test_dataset, 5e-5, 1024, 10, shuffle = True)
torch.save(best_model_wts, '/root/Architectural-Biases-in-Time-Series-Anomaly-Detection/transformer_encoder_FORECASTER_weights.pth')

LR Scheduler: 390 warmup steps, 7810 total steps
batch: 64 | train_loss: 0.2533
batch: 128 | train_loss: 0.2450
batch: 192 | train_loss: 0.2490
batch: 256 | train_loss: 0.2484
batch: 320 | train_loss: 0.2434
batch: 384 | train_loss: 0.2357
batch: 448 | train_loss: 0.2087
batch: 512 | train_loss: 0.1661
batch: 576 | train_loss: 0.1583
batch: 640 | train_loss: 0.1515
batch: 704 | train_loss: 0.1435
batch: 768 | train_loss: 0.1411
|blind_transformer| train = 0.2098 | test= 0.1408 | LR: 4.97e-05
batch: 64 | train_loss: 0.1310
batch: 128 | train_loss: 0.1355
batch: 192 | train_loss: 0.1285
batch: 256 | train_loss: 0.1258
batch: 320 | train_loss: 0.1275
batch: 384 | train_loss: 0.1227
batch: 448 | train_loss: 0.1227
batch: 512 | train_loss: 0.1142
batch: 576 | train_loss: 0.1203
batch: 640 | train_loss: 0.1182
batch: 704 | train_loss: 0.1133
batch: 768 | train_loss: 0.1132
|blind_transformer| train = 0.1245 | test= 0.1124 | LR: 4.70e-05
batch: 64 | train_loss: 0.1105
batch: 128 | train_loss: